# 🔍 Clinisys Silver Layer Reconciliation: Local DuckDB vs AWS Athena Production

This notebook automates the data quality reconciliation between the local **DuckDB** database (`clinisys_all.duckdb`, schema `silver`) and the production **AWS Athena** database (`silver_clinisys_prod`).

### Objectives:
1. **Row Count Audit**: Compare total records per table.
2. **Primary Key Overlap Audit**: Reconcile exact keys present in both environments, only locally, or only in production.
3. **Yearly Breakdown**: Drill down row counts and key matches per calendar year (using mapped date columns).
4. **Newest Record Analysis**: Fetch and show the most recent records present in only one environment (ordered by primary key descending).

### Database Connections:
- **Local**: DuckDB (`clinisys_all.duckdb` -> schema: `silver`)
- **AWS Athena**: PyAthena (`silver_clinisys_prod` database)

In [1]:
import os
import yaml
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Configuration
DUCKDB_PATH = '../../database/clinisys_all.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_clinisys_prod'
CONFIG_PATH = '../../clinisys/column_config.yml'

print("Libraries imported. Configured database paths:")
print(f"  Local DuckDB: {os.path.abspath(DUCKDB_PATH)}")
print(f"  AWS Athena:   {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")

Libraries imported. Configured database paths:
  Local DuckDB: g:\My Drive\projetos_individuais\Huntington\database\clinisys_all.duckdb
  AWS Athena:   silver_clinisys_prod (Region: sa-east-1, Workgroup: datalake-admins)


## 🔌 Connection Helpers
Defining robust wrapper functions to connect, execute queries, and guarantee connection closure.

In [2]:
def run_duck(query):
    """Runs a query on local DuckDB, ensuring connection is closed."""
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        return conn.execute(query).df()
    finally:
        conn.close()

def run_athena(query):
    """Runs a query on AWS Athena, ensuring connection is closed."""
    conn = connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB)
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

# Helper to get columns present in BOTH DuckDB and Athena schemas to prevent schema drift crashes
def get_common_columns(table):
    # 1. Local columns
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        local_cols = set([c[0].lower() for c in conn.execute(f"SELECT * FROM silver.{table} LIMIT 0").description])
    finally:
        conn.close()
    
    # 2. Athena columns
    try:
        prod_df = run_athena(f"SELECT * FROM silver_clinisys_prod.{table} LIMIT 0")
        prod_cols = set([c.lower() for c in prod_df.columns])
    except Exception:
        prod_cols = set()
        
    return list(local_cols & prod_cols)

# Test connections
try:
    duck_ok = run_duck("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ DuckDB Connection: OK")
except Exception as e:
    print(f"❌ DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    athena_ok = run_athena("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False

✅ DuckDB Connection: OK
✅ AWS Athena Connection: OK


## 🗺️ Configuration & Predefined Mappings
Loading primary keys from `column_config.yml` and defining mappings of tables to their primary date columns for yearly breakdowns.

In [3]:
# Load primary keys from configuration
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
primary_keys = config.get('primary_keys', {})

# Map tables to their primary date column (determined from analysis)
TABLE_DATE_COLUMNS = {
    'view_agenda': 'data',
    'view_agendas': 'data',
    'view_congelamentos_embrioes': 'data_congelamento',
    'view_congelamentos_ovulos': 'data_congelamento',
    'view_congelamentos_semen': 'data_congelamento',
    'view_congelamentos_semen_doador': 'data_congelamento',
    'view_descongelamentos_embrioes': 'data_descongelamento',
    'view_descongelamentos_ovulos': 'data_descongelamento',
    'view_embrioes_congelados': 'data_congelamento',
    'view_exames': 'data',
    'view_extrato_atendimentos_central': 'data',
    'view_indicacao_novo': 'data',
    'view_medicamentos_prescricoes': 'data_inicial',
    'view_micromanipulacao': 'data',
    'view_micromanipulacao_oocitos': 'data_procedimento',
    'view_orcamentos': 'data_entrega_orcamento',
    'view_ovulos_congelados': 'data_congelamento',
    'view_procedimentos_financas': 'data_pagamento',
    'view_tratamentos': 'data_procedimento',
    'view_tratamentos_us_anexos': 'data'
}

print(f"Loaded {len(primary_keys)} primary key configs.")
print(f"Predefined date columns for {len(TABLE_DATE_COLUMNS)} tables.")

Loaded 26 primary key configs.
Predefined date columns for 20 tables.


## 🔍 Identify Common Tables
Querying both databases to find matching tables in DuckDB `silver` schema and Athena `silver_clinisys_prod` database.

In [4]:
# Query tables
local_tables_df = run_duck("SELECT table_name FROM information_schema.tables WHERE table_schema = 'silver'")
local_tables = sorted(local_tables_df['table_name'].tolist())

athena_tables_df = run_athena("SHOW TABLES")
athena_tables = sorted(athena_tables_df.iloc[:, 0].tolist())

common_tables = sorted(list(set(local_tables) & set(athena_tables)))
print(f"Found {len(local_tables)} tables in local DuckDB 'silver' schema.")
print(f"Found {len(athena_tables)} tables in Athena '{ATHENA_DB}' database.")
print(f"Common tables to compare ({len(common_tables)}):")
for t in common_tables:
    key_status = f"Key: {primary_keys.get(t)}" if t in primary_keys else "⚠️ No Primary Key in config"
    
    # Schema-aware date column checking
    try:
        cols = get_common_columns(t)
        date_col = TABLE_DATE_COLUMNS.get(t)
        if date_col and date_col.lower() not in cols:
            fallbacks = [c for c in cols if 'data' in c or 'date' in c]
            date_col = fallbacks[0] if fallbacks else None
    except Exception:
        date_col = None
        
    date_status = f"Date: {date_col}" if date_col else "No Date column (All Time)"
    print(f"  - {t:<35} | {key_status:<25} | {date_status}")

Found 26 tables in local DuckDB 'silver' schema.
Found 27 tables in Athena 'silver_clinisys_prod' database.
Common tables to compare (26):
  - view_agenda                         | Key: id                   | Date: data
  - view_agendas                        | Key: id                   | No Date column (All Time)
  - view_congelamentos_embrioes         | Key: id                   | Date: responsavel_recebimento_data
  - view_congelamentos_ovulos           | Key: id                   | Date: responsavel_recebimento_data
  - view_congelamentos_semen            | Key: id                   | Date: responsavel_recebimento_data
  - view_congelamentos_semen_doador     | Key: id                   | Date: responsavel_recebimento_data
  - view_descongelamentos_embrioes      | Key: id                   | No Date column (All Time)
  - view_descongelamentos_ovulos        | Key: id                   | No Date column (All Time)
  - view_embrioes_congelados            | Key: id                   | No

## 📊 Part 1: Row Count & Key Reconciliation Summary
Iterating through all common tables that have defined primary keys to perform count, match, and overlap checks.

In [5]:
summary_data = []

for t in common_tables:
    if t not in primary_keys:
        continue
    
    key = primary_keys[t]
    
    # Row counts
    local_count = run_duck(f"SELECT COUNT(*) as cnt FROM silver.{t}").iloc[0]['cnt']
    prod_count = run_athena(f"SELECT COUNT(*) as cnt FROM silver_clinisys_prod.{t}").iloc[0]['cnt']
    
    # Fetch keys to calculate overlap
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    matched_keys = len(local_keys & prod_keys)
    only_local = len(local_keys - prod_keys)
    only_prod = len(prod_keys - local_keys)
    
    summary_data.append({
        'Table': t,
        'Primary Key': key,
        'Local Rows': local_count,
        'Athena Rows': prod_count,
        'Difference': local_count - prod_count,
        'Match Count': matched_keys,
        'Only in Local (DuckDB)': only_local,
        'Only in Prod (Athena)': only_prod
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    'Local Rows': '{:,}',
    'Athena Rows': '{:,}',
    'Difference': '{:+,}',
    'Match Count': '{:,}',
    'Only in Local (DuckDB)': '{:,}',
    'Only in Prod (Athena)': '{:,}'
}).bar(subset=['Difference'], align='mid', color=['#d65f5f', '#5fba7d'])

,Table,Primary Key,Local Rows,Athena Rows,Difference,Match Count,Only in Local (DuckDB),Only in Prod (Athena)
0,view_agenda,id,"1,138,841","1,139,367",-526,"1,138,436",405,931
1,view_agendas,id,262,262,+0,262,0,0
2,view_congelamentos_embrioes,id,"28,977","28,988",-11,"28,976",1,12
3,view_congelamentos_ovulos,id,"12,487","12,491",-4,"12,487",0,4
4,view_congelamentos_semen,id,"5,759","5,759",+0,"5,759",0,0
5,view_congelamentos_semen_doador,id,"1,978","1,979",-1,"1,978",0,1
6,view_descongelamentos_embrioes,id,"17,954","17,964",-10,"17,954",0,10
7,view_descongelamentos_ovulos,id,"4,065","4,067",-2,"4,065",0,2
8,view_embrioes_congelados,id,"95,852","95,908",-56,"95,849",3,59
9,view_exames,id,"37,150","37,197",-47,"37,150",0,47


## 📅 Part 2: Yearly Breakdown Analysis
Drilling down row counts and key overlaps per year for each table. For tables without defined date columns, counts are grouped under 'N/A'.

In [6]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Schema-aware date column extraction (common to both DBs)
    cols = get_common_columns(t)
    date_col = TABLE_DATE_COLUMNS.get(t)
    if date_col and date_col.lower() not in cols:
        fallbacks = [c for c in cols if 'data' in c or 'date' in c]
        date_col = fallbacks[0] if fallbacks else None
    
    print("=" * 80)
    print(f"📊 Table: {t} (Primary Key: {key} | Date Column: {date_col or 'N/A'})")
    print("=" * 80)
    
    # Fetch keys with date column
    if date_col:
        duck_q = f"SELECT {key} as key_val, {date_col} FROM silver.{t}"
        ath_q = f"SELECT {key} as key_val, {date_col} FROM silver_clinisys_prod.{t}"
        
        local_df = run_duck(duck_q)
        local_df.columns = [c.lower() for c in local_df.columns]
        
        prod_df = run_athena(ath_q)
        prod_df.columns = [c.lower() for c in prod_df.columns]
        
        # Normalize key and date column strings to lowercase
        key_lower = 'key_val'
        date_lower = date_col.lower()
        
        # Parse year safely using Pandas in Python to be 100% format-agnostic
        local_df['record_year'] = pd.to_datetime(local_df[date_lower], dayfirst=True, errors='coerce').dt.year
        local_df['record_year'] = local_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
        
        prod_df['record_year'] = pd.to_datetime(prod_df[date_lower], dayfirst=True, errors='coerce').dt.year
        prod_df['record_year'] = prod_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    else:
        local_df = run_duck(f"SELECT {key} as key_val FROM silver.{t}")
        local_df.columns = [c.lower() for c in local_df.columns]
        local_df['record_year'] = 'N/A'
        
        prod_df = run_athena(f"SELECT {key} as key_val FROM silver_clinisys_prod.{t}")
        prod_df.columns = [c.lower() for c in prod_df.columns]
        prod_df['record_year'] = 'N/A'
        key_lower = 'key_val'
        
    # Unique years
    years = sorted(list(set(local_df['record_year'].dropna().tolist()) | set(prod_df['record_year'].dropna().tolist())))
    
    yearly_summary = []
    for yr in years:
        l_keys = set(local_df[local_df['record_year'] == yr][key_lower].dropna().tolist())
        p_keys = set(prod_df[prod_df['record_year'] == yr][key_lower].dropna().tolist())
        
        matched = len(l_keys & p_keys)
        only_l = len(l_keys - p_keys)
        only_p = len(p_keys - l_keys)
        
        yearly_summary.append({
            'Year': yr,
            'Local Count': len(l_keys),
            'Athena Count': len(p_keys),
            'Matched Count': matched,
            'Only Local': only_l,
            'Only Athena': only_p
        })
        
    yearly_df = pd.DataFrame(yearly_summary)
    display(yearly_df)
    print("\n")

📊 Table: view_agenda (Primary Key: id | Date Column: data)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,1900,29,29,29,0,0
1,1920,1,1,1,0,0
2,1969,736,737,736,0,1
3,1970,2849,2814,2814,35,0
4,1997,1,1,1,0,0
5,2002,1324,1324,1324,0,0
6,2003,5020,5020,5020,0,0
7,2004,8564,8564,8564,0,0
8,2005,10576,10576,10576,0,0
9,2006,10393,10393,10393,0,0




📊 Table: view_agendas (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,262,262,262,0,0




📊 Table: view_congelamentos_embrioes (Primary Key: id | Date Column: responsavel_recebimento_data)


...

,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,2018,685,685,685,0,0
1,2019,19389,19389,19389,0,0
2,2020,17070,17070,17070,0,0
3,2021,22095,22095,22095,0,0
4,2022,23379,23379,23379,0,0
5,2023,45440,45440,45440,0,0
6,2024,61136,61136,61136,0,0
7,2025,100511,100511,100511,0,0
8,2026,83716,84256,83716,0,540




📊 Table: view_unidades (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,11,11,11,0,0




📊 Table: view_usuarios (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,1167,1167,1167,0,0


## 🆕 Part 3: Mismatch Drill-Down — Newest Mismatched Records
Fetching and showcasing up to 5 newest records (ordered by primary key descending) that are exclusive to either the Local database or AWS Athena database.

In [8]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Fetch keys from both
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    only_l_keys = list(local_keys - prod_keys)
    only_p_keys = list(prod_keys - local_keys)
    
    # Sort only keys descending
    only_l_keys.sort(reverse=True)
    only_p_keys.sort(reverse=True)
    
    print("=" * 80)
    print(f"🔍 Mismatch Samples: {t} (Primary Key: {key})")
    print("=" * 80)
    
    # Sample Local only
    if only_l_keys:
        sample_keys = only_l_keys[:]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        local_samples = run_duck(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Local DuckDB (Total: {len(only_l_keys)}):")
        display(local_samples)
    else:
        print("✅ No records found exclusively in Local DuckDB.")
        
    # Sample Athena only
    if only_p_keys:
        sample_keys = only_p_keys[:]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver_clinisys_prod.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        prod_samples = run_athena(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Athena Production (Total: {len(only_p_keys)}):")
        display(prod_samples)
    else:
        print("✅ No records found exclusively in Athena Production.")
    print("\n")

🔍 Mismatch Samples: view_agenda (Primary Key: id)
🆕 Top 405 NEWEST records ONLY found in Local DuckDB (Total: 405):


,id,id_sms,id_laboratorio,data,inicio,termino,recorrente,recorrente_tipo,encaixe,evento,...,identificacao_visual_paciente,usuario_status,indicacao,rd_contato_id,rd_negociacao_id,link_consulta,hash,extraction_timestamp,is_deleted,flag_date_suspect
0,1766422,0,0000046019,2026-08-16,1900-01-01 07:00:00,07:15,0,None,None,765396,...,None,None,None,<NA>,<NA>,None,725794b161f6542bd155be50c18adea7,2026-08-04 19:15:41,0,False
1,1766421,0,0000046019,2026-08-14,1900-01-01 07:00:00,07:15,0,None,None,765396,...,None,None,None,<NA>,<NA>,None,cdd9ff328cad6ad44b77a5789dec177e,2026-08-04 19:15:41,0,False
2,1766283,0,0000031776,2026-08-06,1900-01-01 09:00:00,09:15,0,None,None,71,...,None,None,None,<NA>,<NA>,None,b611e80b6e0012f5e7a06301f26dbc73,2026-08-04 19:15:41,0,False
3,1766282,0,0000031776,2026-08-05,1900-01-01 09:00:00,09:15,0,None,None,71,...,None,None,None,<NA>,<NA>,None,976cd483d13e01d107776ad2f9b499bf,2026-08-04 19:15:41,0,False
4,1766281,0,0000031776,2026-08-04,1900-01-01 09:00:00,09:15,0,None,None,71,...,None,None,None,<NA>,<NA>,None,8f4bfb1393a1e75e5266dd0676f8516a,2026-08-04 19:15:41,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,1486368,0,0000036311,2025-10-14,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,<NA>,<NA>,None,41b3f177766c63aba1b71f4096d374b3,2026-07-18 21:17:49,0,False
401,1486367,0,0000036311,2025-08-19,1900-01-01 07:00:00,07:15,0,None,None,765396,...,None,None,None,<NA>,<NA>,None,11865bf3b5670d6e9b7481536aa75282,2026-07-18 21:17:49,0,False
402,1486366,0,0000036311,2025-08-17,1900-01-01 07:00:00,07:15,0,None,None,765396,...,None,None,None,<NA>,<NA>,None,879cfb3c79de6c7fe84deb9222104e54,2026-07-18 21:17:49,0,False
403,1350347,0,0000023947,2025-01-23,1900-01-01 06:15:00,06:30,0,None,None,71,...,None,None,None,<NA>,<NA>,None,631735cfc1c27db0b289e3ca47f111e3,2026-07-18 21:17:49,0,False


🆕 Top 931 NEWEST records ONLY found in Athena Production (Total: 931):


,id,id_sms,id_laboratorio,data,flag_date_suspect,inicio,termino,recorrente,recorrente_tipo,encaixe,...,tipo_atendimento,acompanhante_paciente,identificacao_visual_paciente,usuario_status,indicacao,rd_contato_id,rd_negociacao_id,link_consulta,bronze_updated_at,_dlt_id
0,1767551,0,None,2026-08-12,False,09:00:00,09:05:00,0,Agendamento normal,None,...,Presencial,None,None,None,None,None,None,None,2026-08-06 02:17:31.056,F8Wx9zdwUfIADg
1,1767550,0,0000042058,2026-07-27,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-06 02:17:31.056,8r/Gh3293L4Xiw
2,1767549,0,0000042058,2026-07-25,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-06 02:17:31.056,cQt/05HtYGrvEg
3,1767548,0,None,2026-08-10,False,06:25:00,06:30:00,0,Agendamento normal,None,...,On-line,None,None,None,None,None,None,None,2026-08-06 02:17:31.056,DEV1/0jg2SwSUw
4,1767547,0,None,2026-08-12,False,07:30:00,07:35:00,0,Agendamento normal,None,...,On-line,None,None,None,None,None,None,None,2026-08-06 02:17:31.056,MZUexF/j2AJB6w
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
926,1766434,0,0000045108,2027-04-13,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,ZgcQXYGFrajiDA
927,1766433,0,0000045108,2027-03-16,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,p71jvtQS4sbUEQ
928,1766432,0,0000045108,2027-02-16,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,buHZyEeDd5dL0w
929,1766431,0,0000045108,2027-01-19,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,LSN+o+gaz/NG8w




🔍 Mismatch Samples: view_agendas (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_congelamentos_embrioes (Primary Key: id)
🆕 Top 1 NEWEST records ONLY found in Local DuckDB (Total: 1):


...



🔍 Mismatch Samples: view_tratamentos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
🆕 Top 31 NEWEST records ONLY found in Athena Production (Total: 31):


,id,prontuario,unidade,idade_esposa,idade_marido,paciente_tratamento,tentativa,data_procedimento,hora_procedimento,tipo_procedimento,...,anomalias_bebe4,observacoes_bebes,observacoes_gerais,usuario_responsavel,responsavel_informacoes,bronze_updated_at,_dlt_id,bmi,previous_et,previous_et_od
0,46276,826643,5.0,41,38,esposa,9,None,None,Ciclo de Congelados,...,None,None,None,5299,646,2026-08-06 02:05:25.669,rNffhCWHe+X00w,21.50,3,0
1,46275,916212,5.0,38,31,esposa,1,None,None,Ciclo a Fresco FIV,...,None,None,None,5299,646,2026-08-06 02:05:25.669,XCILKLV0Fyc2Pg,23.44,0,0
2,46274,879909,5.0,36,42,esposa,4,None,None,Ciclo de Congelados,...,None,None,None,5299,646,2026-08-06 02:05:25.669,QEBQRXqUUDSOFQ,23.44,2,0
3,46273,973681,2.0,39,39,esposa,1,2026-08-09,None,Ciclo a Fresco FIV,...,None,None,Conduta: Coleta + FIV + ZYMOT + PGT-A + Embryo...,3502,389,2026-08-06 02:05:25.669,43zvF334KK6MRw,24.13,0,0
4,46272,974047,2.0,31,34,esposa,1,2026-08-07,None,Ciclo a Fresco FIV,...,None,None,Conduta: Coleta + FIV + ZYMOT + Vitri de emb...,3502,389,2026-08-06 02:05:25.669,WDJLQtfCMvJw+Q,22.50,0,0
5,46271,910411,7.0,36,38,esposa,1,None,None,Ciclo de Congelados,...,None,None,None,9782,219,2026-08-06 02:05:25.669,ETx0T3j6shb+rg,24.98,0,0
6,46270,923173,1.0,38,38,esposa,1,2026-08-10,None,Ciclo de Congelados,...,None,None,None,3540,74,2026-08-06 02:05:25.669,y/fI6wNtL2SJ6Q,21.08,0,0
7,46269,694171,7.0,34,43,esposa,1,2026-08-05,None,Ciclo a Fresco FIV,...,None,None,None,1532,228,2026-08-06 02:05:25.669,XPhGHXqELaEIdQ,29.39,0,0
8,46268,974035,2.0,39,38,esposa,1,2026-08-08,None,Ciclo a Fresco FIV,...,None,None,Conduta: Coleta +ZYMOT + PGTA + Vitri de embr...,3502,389,2026-08-06 02:05:25.669,T8VGhYVpBz2azg,18.51,0,0
9,46267,920109,1.0,40,29,esposa,1,None,None,Ciclo a Fresco FIV,...,None,None,None,6602,8,2026-08-06 02:05:25.669,lji8zFQkhMSHFg,NaN,0,0




🔍 Mismatch Samples: view_tratamentos_us_anexos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
🆕 Top 540 NEWEST records ONLY found in Athena Production (Total: 540):


,id,prontuario,id_tratamento,data,data_us,nome_arquivo,arquivo,bronze_updated_at,_dlt_id
0,381683,912872,0,2026-08-05,05/08/2026,MELYNA VEIGA_6.jpg,ca3888ff669785ff5809f7bacf333b87.jpg,2026-08-06 02:25:14.982,B4N4oyyC038Cig
1,381682,912872,0,2026-08-05,05/08/2026,MELYNA VEIGA_5.jpg,5bf4c726f547bb105645084b709ce079.jpg,2026-08-06 02:25:14.982,SzFwiOIvhjbnIw
2,381681,912872,0,2026-08-05,05/08/2026,MELYNA VEIGA_4.jpg,8c829b339a55a8f82763c3f59bcddf3f.jpg,2026-08-06 02:25:14.982,izn9E3EZJX7jpg
3,381680,912872,0,2026-08-05,05/08/2026,MELYNA VEIGA_3.jpg,100e64589dffdf3d70a9f76d9d87d21a.jpg,2026-08-06 02:25:14.982,IT8hcJKHiUN11w
4,381679,912872,0,2026-08-05,05/08/2026,MELYNA VEIGA_1.jpg,629ba0b661c6a6bcbe05684db740f1a3.jpg,2026-08-06 02:25:14.982,Nqpuy9Gs1s4uBg
...,...,...,...,...,...,...,...,...,...
535,381146,863049,46044,2026-08-05,05/08/2026,CARLA TELLES_4.jpg,75466e93ff1afdfecda26e9f00a5fdf9.jpg,2026-08-06 02:25:14.982,/f2FfiXNGxRnKA
536,381145,863049,46044,2026-08-05,05/08/2026,CARLA TELLES_3.jpg,651ab98f4a090c90d4338c7fd5b686e2.jpg,2026-08-06 02:25:14.982,Ch7k0JUv54WwVQ
537,381144,863049,46044,2026-08-05,05/08/2026,CARLA TELLES_2.jpg,9a2c09a0788581ef58185f7811b43ef5.jpg,2026-08-06 02:25:14.982,mKYN2VRRoAVNsA
538,381143,863049,46044,2026-08-05,05/08/2026,CARLA TELLES_1.jpg,7e9d7a699e1a1c965245798ed11c1d84.jpg,2026-08-06 02:25:14.982,aVhnF0HSrsipvA




🔍 Mismatch Samples: view_unidades (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_usuarios (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


